# DiCE Counterfactual Explanations

This notebook generates counterfactual explanations for the same frozen XGBoost model, saved preprocessing pipeline, exact train/test partitions, and 20 fixed XAI cases used by SHAP and LIME. It does not train, tune, refit, or alter the experiment. Restrictive actionable constraints are preserved even when no valid counterfactual can be found.

## 1–2. Imports and Reproducible Configuration

In [1]:
import json
import random
import time
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from multiprocessing import TimeoutError as ProcessTimeoutError
from pathlib import Path

import dice_ml
import joblib
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score

RANDOM_STATE = 42
TARGET = "TARGET"
FIRST_PASS_COUNTERFACTUALS = 1
OPTIONAL_SECOND_PASS_COUNTERFACTUALS = 3
PER_CASE_TIMEOUT_SECONDS = 30
RUN_OPTIONAL_SECOND_PASS = False
DICE_METHOD = "genetic"
DICE_VERSION = version("dice-ml")
print(f"DiCE version: {DICE_VERSION}")

DiCE version: 0.12


## 3. Load Frozen Artifacts

In [2]:
ARTIFACTS_DIR = Path("../artifacts")
DATA_PATH = Path("../data/raw/application_train.csv")
MODEL_PATH = ARTIFACTS_DIR / "xgboost_credit_model.joblib"
PREPROCESSOR_PATH = ARTIFACTS_DIR / "preprocessor.joblib"
PREPROCESSING_METADATA_PATH = ARTIFACTS_DIR / "preprocessing_metadata.csv"
MODEL_METADATA_PATH = ARTIFACTS_DIR / "model_metadata.json"
PRIMARY_FEATURES_PATH = ARTIFACTS_DIR / "primary_features.json"
TRAIN_INDICES_PATH = ARTIFACTS_DIR / "train_indices.csv"
TEST_INDICES_PATH = ARTIFACTS_DIR / "test_indices.csv"
XAI_CASES_PATH = ARTIFACTS_DIR / "xai_evaluation_cases.csv"

model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)
preprocessing_metadata = pd.read_csv(PREPROCESSING_METADATA_PATH)
with MODEL_METADATA_PATH.open(encoding="utf-8") as file:
    model_metadata = json.load(file)
with PRIMARY_FEATURES_PATH.open(encoding="utf-8") as file:
    modelling_feature_names = json.load(file)
train_indices = pd.read_csv(TRAIN_INDICES_PATH)["row_index"].astype(int)
test_indices = pd.read_csv(TEST_INDICES_PATH)["row_index"].astype(int)
xai_cases = pd.read_csv(XAI_CASES_PATH)
decision_threshold = float(model_metadata["decision_threshold"])
assert hasattr(model, "classes_")
assert np.isclose(decision_threshold, 0.50)
print(f"Frozen model: {MODEL_PATH.name}")
print(f"Frozen preprocessor: {PREPROCESSOR_PATH.name}")
print(f"Decision threshold: {decision_threshold}")

Frozen model: xgboost_credit_model.joblib
Frozen preprocessor: preprocessor.joblib
Decision threshold: 0.5


## 4–5. Reconstruct Exact Data Partitions and Frozen Predictions

In [3]:
ORIGINAL_FEATURES = [
    "NAME_CONTRACT_TYPE", "AMT_INCOME_TOTAL", "AMT_CREDIT",
    "AMT_ANNUITY", "AMT_GOODS_PRICE", "DAYS_EMPLOYED",
    "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE", "CNT_FAM_MEMBERS",
    "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]
raw_df = pd.read_csv(DATA_PATH)
modelling_df = raw_df[ORIGINAL_FEATURES + [TARGET]].copy()
employment_days = modelling_df["DAYS_EMPLOYED"].mask(modelling_df["DAYS_EMPLOYED"].eq(365243), np.nan)
modelling_df["EMPLOYMENT_YEARS"] = employment_days.abs().div(365.25)
modelling_df.drop(columns=["DAYS_EMPLOYED"], inplace=True)

def safe_ratio(numerator, denominator):
    return numerator.div(denominator.mask(denominator.eq(0), np.nan))

def recalculate_ratios(frame):
    frame = frame.copy()
    frame["CREDIT_INCOME_RATIO"] = safe_ratio(frame["AMT_CREDIT"], frame["AMT_INCOME_TOTAL"])
    frame["ANNUITY_INCOME_RATIO"] = safe_ratio(frame["AMT_ANNUITY"], frame["AMT_INCOME_TOTAL"])
    frame["CREDIT_ANNUITY_RATIO"] = safe_ratio(frame["AMT_CREDIT"], frame["AMT_ANNUITY"])
    with pd.option_context("future.no_silent_downcasting", True):
        cleaned = frame.replace([np.inf, -np.inf], np.nan)
    return cleaned.infer_objects(copy=False)

modelling_df = recalculate_ratios(modelling_df)
assert modelling_df.columns.drop(TARGET).tolist() == modelling_feature_names
X_train_raw = modelling_df.loc[train_indices, modelling_feature_names].copy()
X_test_raw = modelling_df.loc[test_indices, modelling_feature_names].copy()
y_train = modelling_df.loc[train_indices, TARGET].copy()
y_test = modelling_df.loc[test_indices, TARGET].copy()
X_test_processed = preprocessor.transform(X_test_raw)
test_probabilities = model.predict_proba(X_test_processed)[:, 1]
test_predictions = (test_probabilities >= decision_threshold).astype(int)
reproduced_metrics = {
    "roc_auc": roc_auc_score(y_test, test_probabilities),
    "pr_auc": average_precision_score(y_test, test_probabilities),
    "precision_positive": precision_score(y_test, test_predictions, zero_division=0),
    "recall_positive": recall_score(y_test, test_predictions, zero_division=0),
    "f1_positive": f1_score(y_test, test_predictions, zero_division=0),
}
print(f"Training rows: {len(X_train_raw)}; TARGET 1: {y_train.mean() * 100:.4f}%")
print(f"Test rows: {len(X_test_raw)}; TARGET 1: {y_test.mean() * 100:.4f}%")
for metric, value in reproduced_metrics.items():
    print(f"{metric}: {value:.6f}")
    if metric in model_metadata and not np.isclose(value, float(model_metadata[metric]), atol=1e-6):
        raise RuntimeError(f"Frozen metric reproduction failed for {metric}.")
assert len(X_train_raw) == 246008 and len(X_test_raw) == 61503
assert X_test_processed.shape[1] == 28

Training rows: 246008; TARGET 1: 8.0729%
Test rows: 61503; TARGET 1: 8.0728%
roc_auc: 0.692518
pr_auc: 0.167144
precision_positive: 0.138796
recall_positive: 0.621551
f1_positive: 0.226920


## 6. Load and Verify the Same 20 XAI Cases

In [4]:
assert len(xai_cases) == 20
assert xai_cases["case_id"].tolist() == [f"XAI_{number:03d}" for number in range(1, 21)]
assert xai_cases["row_index"].isin(test_indices).all()
test_position_by_index = pd.Series(np.arange(len(test_indices)), index=test_indices.to_numpy())
case_positions = test_position_by_index.loc[xai_cases["row_index"]].to_numpy()
assert np.allclose(test_probabilities[case_positions], xai_cases["predicted_probability"], atol=1e-6)
assert np.array_equal(test_predictions[case_positions], xai_cases["predicted_class"].to_numpy())
assert np.array_equal(y_test.iloc[case_positions].to_numpy(), xai_cases["true_target"].to_numpy())
display(xai_cases)

,case_id,row_index,true_target,predicted_class,predicted_probability,case_type
0,XAI_001,288644,0,1,0.871334,high-confidence positive
1,XAI_002,234361,0,1,0.861796,high-confidence positive
2,XAI_003,25106,1,1,0.851671,high-confidence positive
3,XAI_004,148440,0,1,0.845883,high-confidence positive
4,XAI_005,267529,0,1,0.844541,high-confidence positive
5,XAI_006,259190,0,1,0.500000,borderline positive
6,XAI_007,125832,0,1,0.500008,borderline positive
7,XAI_008,3508,0,1,0.500023,borderline positive
8,XAI_009,264908,0,1,0.500030,borderline positive
9,XAI_010,253960,0,1,0.500040,borderline positive


## 7. Human-Readable Feature Names

In [5]:
DISPLAY_NAMES = {
    "NAME_CONTRACT_TYPE": "Contract type", "AMT_INCOME_TOTAL": "Annual income",
    "AMT_CREDIT": "Requested credit amount", "AMT_ANNUITY": "Annuity amount",
    "AMT_GOODS_PRICE": "Goods price", "CNT_FAM_MEMBERS": "Family members",
    "EMPLOYMENT_YEARS": "Employment duration", "NAME_INCOME_TYPE": "Income type",
    "NAME_HOUSING_TYPE": "Housing type",
    "AMT_REQ_CREDIT_BUREAU_MON": "Credit enquiries in previous month",
    "AMT_REQ_CREDIT_BUREAU_QRT": "Credit enquiries in previous quarter",
    "AMT_REQ_CREDIT_BUREAU_YEAR": "Credit enquiries in previous year",
    "CREDIT_INCOME_RATIO": "Credit-to-income ratio",
    "ANNUITY_INCOME_RATIO": "Annuity-to-income ratio",
    "CREDIT_ANNUITY_RATIO": "Credit-to-annuity ratio",
}
display(pd.Series(DISPLAY_NAMES, name="display_name").rename_axis("feature").to_frame())

,display_name
feature,
NAME_CONTRACT_TYPE,Contract type
AMT_INCOME_TOTAL,Annual income
AMT_CREDIT,Requested credit amount
AMT_ANNUITY,Annuity amount
AMT_GOODS_PRICE,Goods price
CNT_FAM_MEMBERS,Family members
EMPLOYMENT_YEARS,Employment duration
NAME_INCOME_TYPE,Income type
NAME_HOUSING_TYPE,Housing type


## 8–9. Conservative Actionability and Training-Based Permitted Ranges

The primary experiment varies only requested credit, annuity, and goods price. Income is excluded because treating short-term income change as actionable requires a separate methodological justification. Immutable, historical, categorical, bureau-history, and derived features remain locked. Derived ratios are recomputed automatically and never counted as independent actions. Permitted ranges use the 1st and 99th training percentiles to reduce implausible extremes without consulting test data.

In [6]:
features_to_vary = ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
DERIVED_FEATURES = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO"]
CONTINUOUS_FEATURES = X_train_raw.select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_FEATURES = X_train_raw.select_dtypes(exclude=np.number).columns.tolist()
permitted_ranges = {}
training_iqr = {}
for feature in features_to_vary:
    valid_values = X_train_raw[feature].replace([np.inf, -np.inf], np.nan).dropna()
    lower, upper = valid_values.quantile([0.01, 0.99]).astype(float)
    permitted_ranges[feature] = [max(0.0, lower), upper]
    iqr = float(valid_values.quantile(0.75) - valid_values.quantile(0.25))
    training_iqr[feature] = iqr
    assert permitted_ranges[feature][0] < permitted_ranges[feature][1] and iqr > 0
display(pd.DataFrame(permitted_ranges, index=["lower", "upper"]).T)

,lower,upper
AMT_CREDIT,76410.0,1862802.0
AMT_ANNUITY,6187.5,70033.5
AMT_GOODS_PRICE,67500.0,1800000.0


## 10. Build and Validate the DiCE Prediction Interface

In [7]:
class FrozenPipelineModel:
    def __init__(self, frozen_preprocessor, frozen_model):
        self.preprocessor = frozen_preprocessor
        self.model = frozen_model
        self.classes_ = frozen_model.classes_

    def _prepare(self, samples):
        frame = samples.copy() if isinstance(samples, pd.DataFrame) else pd.DataFrame(samples, columns=modelling_feature_names)
        frame = frame[modelling_feature_names].copy()
        return recalculate_ratios(frame)

    def predict_proba(self, samples):
        frame = self._prepare(samples)
        return self.model.predict_proba(self.preprocessor.transform(frame))

    def predict(self, samples):
        probabilities = self.predict_proba(samples)[:, 1]
        return (probabilities >= decision_threshold).astype(int)

dice_prediction_model = FrozenPipelineModel(preprocessor, model)
case_rows = X_test_raw.loc[xai_cases["row_index"], modelling_feature_names]
wrapper_probabilities = dice_prediction_model.predict_proba(case_rows)[:, 1]
wrapper_max_difference = float(np.max(np.abs(wrapper_probabilities - xai_cases["predicted_probability"])))
print(f"DiCE wrapper maximum probability difference: {wrapper_max_difference:.12g}")
assert np.allclose(wrapper_probabilities, xai_cases["predicted_probability"], atol=1e-7)

DiCE wrapper maximum probability difference: 1.95549011428e-08


## 11. Configure DiCE with Training Data Only

Genetic search constructs a KD-tree and therefore requires a finite reference matrix. A DiCE-only training reference copy is filled with training-partition medians and modes, then all ratios are recomputed. This does not fit or alter the frozen preprocessor; every model query still passes through the saved preprocessing pipeline.

In [8]:
dice_training_features = X_train_raw[modelling_feature_names].copy()
non_derived_numeric_features = [feature for feature in CONTINUOUS_FEATURES if feature not in DERIVED_FEATURES]
dice_numeric_medians = X_train_raw[non_derived_numeric_features].median()
dice_categorical_modes = X_train_raw[CATEGORICAL_FEATURES].mode(dropna=True).iloc[0]
dice_training_features[non_derived_numeric_features] = dice_training_features[non_derived_numeric_features].fillna(dice_numeric_medians)
dice_training_features[CATEGORICAL_FEATURES] = dice_training_features[CATEGORICAL_FEATURES].fillna(dice_categorical_modes)
dice_training_features = recalculate_ratios(dice_training_features)
dice_training_data = dice_training_features.copy()
dice_training_data[TARGET] = y_train.to_numpy()
dice_reference_numeric = dice_training_features.select_dtypes(include=np.number).to_numpy()
assert int(dice_training_features.isna().sum().sum()) == 0
assert np.isfinite(dice_reference_numeric).all()
dice_data = dice_ml.Data(
    dataframe=dice_training_data, continuous_features=CONTINUOUS_FEATURES,
    outcome_name=TARGET, permitted_range=permitted_ranges,
)
dice_model = dice_ml.Model(model=dice_prediction_model, backend="sklearn", model_type="classifier")
dice_explainer = dice_ml.Dice(dice_data, dice_model, method=DICE_METHOD)
print(f"Training-only DiCE reference rows: {len(dice_training_data)}")
print(f"Features permitted to vary: {features_to_vary}")

Training-only DiCE reference rows: 246008
Features permitted to vary: ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']


## 12. Bounded Single-Case Diagnostic — XAI_005

The diagnostic must complete before the 20-case first pass is started. Generation runs in an isolated `loky` worker process with a 30-second timeout. This avoids unsafe thread cancellation; a timed-out or serialization-failing worker is discarded and reported without changing the search constraints.

In [9]:
def isolated_dice_search(explainer, query, desired_class, total_counterfactuals):
    try:
        np.random.seed(RANDOM_STATE)
        random.seed(RANDOM_STATE)
        result = explainer.generate_counterfactuals(
            query, total_CFs=total_counterfactuals, desired_class=desired_class,
            features_to_vary=features_to_vary, permitted_range=permitted_ranges,
            verbose=False,
        )
        generated = result.cf_examples_list[0].final_cfs_df
        return {"generated": generated, "exception_class": "", "error_message": "", "traceback": ""}
    except Exception as error:
        return {
            "generated": None, "exception_class": type(error).__name__,
            "error_message": str(error), "traceback": traceback.format_exc(),
        }

def bounded_dice_search(query, desired_class, total_counterfactuals, timeout_seconds=30):
    try:
        outcome = Parallel(n_jobs=2, backend="loky", timeout=timeout_seconds)(
            [delayed(isolated_dice_search)(dice_explainer, query, desired_class, total_counterfactuals)]
        )[0]
        generated = outcome["generated"]
        if outcome["exception_class"]:
            if "No Counterfactuals found" in outcome["error_message"]:
                return None, "no valid counterfactual found", outcome["error_message"], outcome["exception_class"], outcome["traceback"]
            return None, "error", outcome["error_message"], outcome["exception_class"], outcome["traceback"]
        if generated is None or generated.empty:
            return None, "no valid counterfactual found", "", "", ""
        return generated, "success", "", "", ""
    except ProcessTimeoutError:
        return None, "timeout", f"counterfactual search exceeded {timeout_seconds} seconds", "ProcessTimeoutError", traceback.format_exc()
    except Exception as error:
        message = str(error)
        if "No Counterfactuals found" in message:
            return None, "no valid counterfactual found", message, type(error).__name__, traceback.format_exc()
        return None, "error", message, type(error).__name__, traceback.format_exc()

diagnostic_case = xai_cases.loc[xai_cases["case_id"].eq("XAI_005")].iloc[0]
diagnostic_query = X_test_raw.loc[[diagnostic_case["row_index"]], modelling_feature_names].copy()
diagnostic_desired_class = 1 - int(diagnostic_case["predicted_class"])
diagnostic_start_time = datetime.now(timezone.utc)
diagnostic_timer = time.perf_counter()
print(f"Query shape: {diagnostic_query.shape}")
print(f"Query columns: {diagnostic_query.columns.tolist()}")
display(diagnostic_query.dtypes.rename("query_dtype").to_frame())
print(f"Query NaN count: {int(diagnostic_query.isna().sum().sum())}")
diagnostic_numeric = diagnostic_query.select_dtypes(include=np.number).to_numpy()
print(f"Query infinite-value count: {int(np.isinf(diagnostic_numeric).sum())}")
print(f"Categorical values: {diagnostic_query[CATEGORICAL_FEATURES].iloc[0].to_dict()}")
display(diagnostic_query[["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"] + DERIVED_FEATURES])
dtype_comparison = pd.DataFrame({
    "training_dtype": dice_training_data[modelling_feature_names].dtypes.astype(str),
    "query_dtype": diagnostic_query.dtypes.astype(str),
})
display(dtype_comparison)
assert diagnostic_query.columns.tolist() == dice_training_data.columns.drop(TARGET).tolist()

diagnostic_generated, diagnostic_status, diagnostic_error, diagnostic_exception_class, diagnostic_traceback = bounded_dice_search(
    diagnostic_query, diagnostic_desired_class, FIRST_PASS_COUNTERFACTUALS
)
diagnostic_elapsed = time.perf_counter() - diagnostic_timer
diagnostic_end_time = datetime.now(timezone.utc)
diagnostic_summary = pd.DataFrame([{
    "case_id": diagnostic_case["case_id"],
    "original_probability": diagnostic_case["predicted_probability"],
    "original_class": int(diagnostic_case["predicted_class"]),
    "desired_class": diagnostic_desired_class,
    "generation_start_time": diagnostic_start_time.isoformat(),
    "generation_end_time": diagnostic_end_time.isoformat(),
    "elapsed_seconds": diagnostic_elapsed,
    "dice_method": DICE_METHOD,
    "counterfactuals_requested": FIRST_PASS_COUNTERFACTUALS,
    "counterfactuals_found": 0 if diagnostic_generated is None else len(diagnostic_generated),
    "generation_status": diagnostic_status,
    "exception_class": diagnostic_exception_class,
    "error_message": diagnostic_error,
}])
display(diagnostic_summary)
print(f"features_to_vary: {features_to_vary}")
print(f"permitted_range: {permitted_ranges}")
print(f"DiCE method: {DICE_METHOD}")
if diagnostic_traceback:
    print("Full diagnostic traceback:")
    print(diagnostic_traceback)
print(f"XAI_005 diagnostic completed in {diagnostic_elapsed:.2f}s: {diagnostic_status}")
if diagnostic_generated is not None and not diagnostic_generated.empty:
    diagnostic_cf = diagnostic_generated.drop(columns=[TARGET], errors="ignore")[modelling_feature_names].copy()
    diagnostic_cf = recalculate_ratios(diagnostic_cf)
    diagnostic_cf_probability = float(dice_prediction_model.predict_proba(diagnostic_cf)[:, 1][0])
    diagnostic_cf_class = int(diagnostic_cf_probability >= decision_threshold)
    diagnostic_valid = diagnostic_cf_class == diagnostic_desired_class
    diagnostic_changes = []
    for feature in features_to_vary:
        original_value = diagnostic_query.iloc[0][feature]
        counterfactual_value = diagnostic_cf.iloc[0][feature]
        if not np.isclose(float(original_value), float(counterfactual_value), equal_nan=True):
            diagnostic_changes.append({
                "feature": feature, "original_value": original_value,
                "counterfactual_value": counterfactual_value,
            })
    diagnostic_validation = pd.DataFrame([{
        "case_id": diagnostic_case["case_id"],
        "original_probability": float(diagnostic_case["predicted_probability"]),
        "counterfactual_probability": diagnostic_cf_probability,
        "independently_reproduced_counterfactual_class": diagnostic_cf_class,
        "desired_class": diagnostic_desired_class,
        "valid_counterfactual": diagnostic_valid,
    }])
    display(diagnostic_validation)
    display(pd.DataFrame(diagnostic_changes))

Query shape: (1, 15)
Query columns: ['NAME_CONTRACT_TYPE', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_INCOME_TYPE', 'NAME_HOUSING_TYPE', 'CNT_FAM_MEMBERS', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'EMPLOYMENT_YEARS', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_ANNUITY_RATIO']


,query_dtype
NAME_CONTRACT_TYPE,object
AMT_INCOME_TOTAL,float64
AMT_CREDIT,float64
AMT_ANNUITY,float64
AMT_GOODS_PRICE,float64
NAME_INCOME_TYPE,object
NAME_HOUSING_TYPE,object
CNT_FAM_MEMBERS,float64
AMT_REQ_CREDIT_BUREAU_MON,float64
AMT_REQ_CREDIT_BUREAU_QRT,float64


Query NaN count: 0
Query infinite-value count: 0
Categorical values: {'NAME_CONTRACT_TYPE': 'Cash loans', 'NAME_INCOME_TYPE': 'Working', 'NAME_HOUSING_TYPE': 'House / apartment'}


,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_ANNUITY_RATIO
267529,545040.0,26640.0,450000.0,3.364444,0.164444,20.459459


,training_dtype,query_dtype
NAME_CONTRACT_TYPE,object,object
AMT_INCOME_TOTAL,float64,float64
AMT_CREDIT,float64,float64
AMT_ANNUITY,float64,float64
AMT_GOODS_PRICE,float64,float64
NAME_INCOME_TYPE,object,object
NAME_HOUSING_TYPE,object,object
CNT_FAM_MEMBERS,float64,float64
AMT_REQ_CREDIT_BUREAU_MON,float64,float64
AMT_REQ_CREDIT_BUREAU_QRT,float64,float64


,case_id,original_probability,original_class,desired_class,generation_start_time,generation_end_time,elapsed_seconds,dice_method,counterfactuals_requested,counterfactuals_found,generation_status,exception_class,error_message
0,XAI_005,0.844541,1,0,2026-08-21T20:22:47.942110+00:00,2026-08-21T20:23:06.373996+00:00,18.431764,genetic,1,1,success,,


features_to_vary: ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']
permitted_range: {'AMT_CREDIT': [76410.0, 1862802.0], 'AMT_ANNUITY': [6187.5, 70033.5], 'AMT_GOODS_PRICE': [67500.0, 1800000.0]}
DiCE method: genetic
XAI_005 diagnostic completed in 18.43s: success


,case_id,original_probability,counterfactual_probability,independently_reproduced_counterfactual_class,desired_class,valid_counterfactual
0,XAI_005,0.844541,0.418978,0,0,True


,feature,original_value,counterfactual_value
0,AMT_ANNUITY,26640.0,6187.5


## 13. Bounded 20-Case First Pass

Run this cell only after the XAI_005 diagnostic above completes. It requests one counterfactual per case and records success, legitimate no-counterfactual results, timeouts, and errors. `KeyboardInterrupt` remains separate so a manual interruption preserves the already completed in-memory log.

In [10]:
RUN_FULL_FIRST_PASS = False
counterfactual_records = []
generation_errors = {}
first_pass_generated = {}
first_pass_records = []
try:
    for case in (xai_cases.itertuples(index=False) if RUN_FULL_FIRST_PASS else []):
        print(f"Generating {case.case_id}...")
        query = X_test_raw.loc[[case.row_index], modelling_feature_names].copy()
        desired_class = 1 - int(case.predicted_class)
        started_at = datetime.now(timezone.utc)
        timer = time.perf_counter()
        generated, status, error_message, exception_class, case_traceback = bounded_dice_search(
            query, desired_class, FIRST_PASS_COUNTERFACTUALS
        )
        elapsed = time.perf_counter() - timer
        found = 0 if generated is None else len(generated)
        first_pass_records.append({
            "case_id": case.case_id, "original_probability": float(case.predicted_probability),
            "original_class": int(case.predicted_class), "desired_class": desired_class,
            "start_time": started_at.isoformat(), "elapsed_seconds": elapsed,
            "counterfactuals_requested": FIRST_PASS_COUNTERFACTUALS,
            "counterfactuals_found": found, "generation_status": status,
            "error_message": error_message, "exception_class": exception_class,
        })
        if generated is not None:
            first_pass_generated[case.case_id] = generated
        else:
            generation_errors[case.case_id] = error_message or status
        print(f"Finished {case.case_id} in {elapsed:.2f}s: {status}")
except KeyboardInterrupt:
    print("First pass interrupted by user; completed case results have been preserved.")
    completed_case_ids = {record["case_id"] for record in first_pass_records}
    for remaining in xai_cases.loc[~xai_cases["case_id"].isin(completed_case_ids)].itertuples(index=False):
        first_pass_records.append({
            "case_id": remaining.case_id, "original_probability": float(remaining.predicted_probability),
            "original_class": int(remaining.predicted_class), "desired_class": 1 - int(remaining.predicted_class),
            "start_time": None, "elapsed_seconds": np.nan,
            "counterfactuals_requested": FIRST_PASS_COUNTERFACTUALS,
            "counterfactuals_found": 0, "generation_status": "not run - interrupted",
            "error_message": "manual KeyboardInterrupt before this case was started",
        })

FIRST_PASS_PATH = ARTIFACTS_DIR / "dice_generation_first_pass.csv"
if RUN_FULL_FIRST_PASS:
    dice_generation_first_pass = pd.DataFrame(first_pass_records)
    dice_generation_first_pass.to_csv(FIRST_PASS_PATH, index=False)
else:
    dice_generation_first_pass = pd.read_csv(FIRST_PASS_PATH, keep_default_na=False)
    print("Full first pass skipped; loaded existing dice_generation_first_pass.csv.")
display(dice_generation_first_pass)
assert len(dice_generation_first_pass) == len(xai_cases)
assert dice_generation_first_pass["case_id"].tolist() == xai_cases["case_id"].tolist()

Full first pass skipped; loaded existing dice_generation_first_pass.csv.


,case_id,original_probability,original_class,desired_class,start_time,elapsed_seconds,counterfactuals_requested,counterfactuals_found,generation_status,error_message,exception_class
0,XAI_001,0.871334,1,0,2026-08-20T22:30:24.344109+00:00,7.191677,1,1,success,,
1,XAI_002,0.861796,1,0,2026-08-20T22:30:31.541666+00:00,13.077958,1,1,success,,
2,XAI_003,0.851671,1,0,2026-08-20T22:30:44.627666+00:00,6.179686,1,1,success,,
3,XAI_004,0.845883,1,0,2026-08-20T22:30:50.816534+00:00,6.529156,1,1,success,,
4,XAI_005,0.844541,1,0,2026-08-20T22:30:57.353539+00:00,6.720849,1,1,success,,
5,XAI_006,0.500000,1,0,2026-08-20T22:31:04.082499+00:00,7.400179,1,1,success,,
6,XAI_007,0.500008,1,0,2026-08-20T22:31:11.489745+00:00,0.333317,1,0,error,The query instance(s) should not have any miss...,UserConfigValidationException
7,XAI_008,0.500023,1,0,2026-08-20T22:31:11.827405+00:00,30.161798,1,0,timeout,counterfactual search exceeded 30 seconds,ProcessTimeoutError
8,XAI_009,0.500030,1,0,2026-08-20T22:31:41.996161+00:00,11.757023,1,1,success,,
9,XAI_010,0.500040,1,0,2026-08-20T22:31:53.761924+00:00,30.099692,1,0,timeout,counterfactual search exceeded 30 seconds,ProcessTimeoutError


## Optional Second Pass for First-Pass Successes

This pass is disabled by default. If explicitly enabled, only first-pass successes request up to three counterfactuals; known no-CF and timeout cases are not rerun.

In [11]:
generation_results = dict(first_pass_generated)
if not RUN_FULL_FIRST_PASS:
    persisted_recovery_path = ARTIFACTS_DIR / "dice_recovered_first_pass_counterfactuals.csv"
    if not persisted_recovery_path.exists():
        raise FileNotFoundError(
            "Generation is disabled and the recovered counterfactual artifact is missing."
        )
    persisted_recovery = pd.read_csv(persisted_recovery_path, keep_default_na=False)
    assert len(persisted_recovery) == 8
    counterfactual_records = [
        {
            "case_id": row.case_id,
            "counterfactual_id": row.counterfactual_id,
            "original_probability": float(row.original_probability),
            "original_class": int(row.original_class),
            "counterfactual_probability": float(row.counterfactual_probability),
            "counterfactual_class": int(row.counterfactual_class),
            "desired_class": int(row.desired_class),
            "valid_counterfactual": str(row.valid_counterfactual).lower() in {"true", "1", "1.0"},
            "counterfactual_row": {
                feature: getattr(row, feature) for feature in modelling_feature_names
            },
        }
        for row in persisted_recovery.itertuples(index=False)
    ]
    print("Loaded 8 persisted recovered counterfactuals for downstream analysis.")
if RUN_OPTIONAL_SECOND_PASS:
    successful_case_ids = dice_generation_first_pass.loc[
        dice_generation_first_pass["generation_status"].eq("success"), "case_id"
    ].tolist()
    for case_id in successful_case_ids:
        case = xai_cases.loc[xai_cases["case_id"].eq(case_id)].iloc[0]
        query = X_test_raw.loc[[case["row_index"]], modelling_feature_names].copy()
        generated, status, error_message, exception_class, case_traceback = bounded_dice_search(
            query, 1 - int(case["predicted_class"]),
            OPTIONAL_SECOND_PASS_COUNTERFACTUALS,
        )
        if status == "success":
            generation_results[case_id] = generated
        else:
            print(f"Optional second pass retained first-pass result for {case_id}: {status}")

for case_id, generated in generation_results.items():
    case = xai_cases.loc[xai_cases["case_id"].eq(case_id)].iloc[0]
    desired_class = 1 - int(case["predicted_class"])
    generated = generated.drop(columns=[TARGET], errors="ignore")[modelling_feature_names]
    generated = recalculate_ratios(generated)
    independent_probabilities = dice_prediction_model.predict_proba(generated)[:, 1]
    independent_classes = (independent_probabilities >= decision_threshold).astype(int)
    for number, (_, generated_row) in enumerate(generated.iterrows(), start=1):
        counterfactual_records.append({
            "case_id": case_id, "counterfactual_id": f"{case_id}_CF{number:02d}",
            "original_probability": float(case["predicted_probability"]),
            "original_class": int(case["predicted_class"]),
            "counterfactual_probability": float(independent_probabilities[number - 1]),
            "counterfactual_class": int(independent_classes[number - 1]),
            "desired_class": desired_class,
            "valid_counterfactual": bool(independent_classes[number - 1] == desired_class),
            "counterfactual_row": generated_row.to_dict(),
        })
print(f"Counterfactual rows ready for independent validation: {len(counterfactual_records)}")

Counterfactual rows ready for independent validation: 0


## 14–17. Feature Changes, Sparsity, Proximity, Plausibility, and Actionability

In [12]:
DERIVED_TOLERANCE = 1e-8
validated_records = []
change_records = []
for record in counterfactual_records:
    case = xai_cases.loc[xai_cases["case_id"].eq(record["case_id"])].iloc[0]
    original_row = X_test_raw.loc[case["row_index"], modelling_feature_names]
    counterfactual_row = pd.Series(record.pop("counterfactual_row"))
    direct_changed = []
    derived_changed = []
    proximity_score = 0.0
    failure_reasons = []
    for feature in modelling_feature_names:
        original_value, new_value = original_row[feature], counterfactual_row[feature]
        if feature in CATEGORICAL_FEATURES:
            changed = str(original_value) != str(new_value)
            absolute_change = np.nan
            relative_change = np.nan
        else:
            changed = not np.isclose(float(original_value), float(new_value), equal_nan=True)
            absolute_change = abs(float(new_value) - float(original_value)) if pd.notna(original_value) and pd.notna(new_value) else np.nan
            relative_change = absolute_change / abs(float(original_value)) if pd.notna(absolute_change) and float(original_value) != 0 else np.nan
        if changed:
            if feature in DERIVED_FEATURES:
                derived_changed.append(feature)
            else:
                direct_changed.append(feature)
            change_records.append({
                "case_id": record["case_id"], "counterfactual_id": record["counterfactual_id"],
                "feature": feature, "display_feature": DISPLAY_NAMES[feature],
                "original_value": original_value, "counterfactual_value": new_value,
                "absolute_change": absolute_change, "relative_change": relative_change,
                "changed": True, "direct_action": feature not in DERIVED_FEATURES,
            })
    for feature in direct_changed:
        if feature not in features_to_vary:
            failure_reasons.append(f"non-permitted feature changed: {feature}")
    for feature in features_to_vary:
        value = float(counterfactual_row[feature])
        lower, upper = permitted_ranges[feature]
        if not (lower <= value <= upper) or value < 0:
            failure_reasons.append(f"{feature} outside permitted range")
        if feature in direct_changed:
            proximity_score += abs(value - float(original_row[feature])) / training_iqr[feature]
    expected_ratios = recalculate_ratios(pd.DataFrame([counterfactual_row]))[DERIVED_FEATURES].iloc[0]
    if not np.allclose(counterfactual_row[DERIVED_FEATURES].astype(float), expected_ratios.astype(float), atol=DERIVED_TOLERANCE, equal_nan=True):
        failure_reasons.append("derived ratios inconsistent")
    actionability_pass = set(direct_changed).issubset(features_to_vary)
    plausibility_pass = len(failure_reasons) == 0 and record["valid_counterfactual"]
    validated_records.append({
        **record, "number_of_changed_features": len(direct_changed),
        "automatically_recomputed_derived_features": "; ".join(derived_changed),
        "proximity_score": float(proximity_score),
        "permitted_ranges_respected": not any("outside permitted range" in reason for reason in failure_reasons),
        "derived_ratios_consistent": "derived ratios inconsistent" not in failure_reasons,
        "actionability_pass": bool(actionability_pass),
        "plausibility_pass": bool(plausibility_pass),
        "failure_reason": "; ".join(failure_reasons),
    })

counterfactual_columns = [
    "case_id", "counterfactual_id", "original_probability", "original_class",
    "counterfactual_probability", "counterfactual_class", "desired_class",
    "valid_counterfactual", "number_of_changed_features",
    "automatically_recomputed_derived_features", "proximity_score",
    "permitted_ranges_respected", "derived_ratios_consistent",
    "actionability_pass", "plausibility_pass", "failure_reason",
]
change_columns = [
    "case_id", "counterfactual_id", "feature", "display_feature",
    "original_value", "counterfactual_value", "absolute_change",
    "relative_change", "changed", "direct_action",
]
dice_counterfactuals = pd.DataFrame(validated_records, columns=counterfactual_columns)
dice_feature_changes = pd.DataFrame(change_records, columns=change_columns)
valid_counterfactuals = dice_counterfactuals.loc[
    dice_counterfactuals["valid_counterfactual"] & dice_counterfactuals["plausibility_pass"] & dice_counterfactuals["actionability_pass"]
].copy()
dice_sparsity_metrics = valid_counterfactuals[["case_id", "counterfactual_id", "number_of_changed_features"]].copy()
dice_proximity_metrics = valid_counterfactuals[["case_id", "counterfactual_id", "proximity_score"]].copy()
display(dice_sparsity_metrics["number_of_changed_features"].agg(["mean", "median", "min", "max"]).to_frame("value"))

,value
mean,NaN
median,NaN
min,NaN
max,NaN


## 18. Deterministic Plain-English Counterfactual Explanations

In [13]:
plain_english_records = []
for counterfactual in valid_counterfactuals.itertuples(index=False):
    changes = dice_feature_changes.loc[
        dice_feature_changes["counterfactual_id"].eq(counterfactual.counterfactual_id) & dice_feature_changes["direct_action"]
    ]
    phrases = [f"{row.display_feature.lower()} changed from {row.original_value} to {row.counterfactual_value}" for row in changes.itertuples()]
    change_text = ", and ".join(phrases)
    destination = "higher-risk" if counterfactual.counterfactual_class == 1 else "lower-risk"
    explanation = (
        f"The model assigns the original application a {counterfactual.original_probability:.1%} higher-risk probability. "
        f"Under a hypothetical input where {change_text}, the model produces a "
        f"{counterfactual.counterfactual_probability:.1%} higher-risk probability and predicts the {destination} class. "
        "This counterfactual describes model behaviour and is not financial advice or a causal claim."
    )
    plain_english_records.append({"case_id": counterfactual.case_id, "counterfactual_id": counterfactual.counterfactual_id, "plain_english_explanation": explanation})
dice_plain_english = pd.DataFrame(plain_english_records)
display(dice_plain_english.head())

""


## 19. Representative Counterfactual Visualisation

In [14]:
representative_pool = valid_counterfactuals.loc[valid_counterfactuals["case_id"].eq("XAI_005")]
if representative_pool.empty:
    representative_pool = valid_counterfactuals
PLOTS_DIR = ARTIFACTS_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DICE_PLOT_PATH = PLOTS_DIR / "dice_representative_counterfactual.png"
if not representative_pool.empty:
    representative = representative_pool.sort_values(["number_of_changed_features", "proximity_score"]).iloc[0]
    representative_changes = dice_feature_changes.loc[
        dice_feature_changes["counterfactual_id"].eq(representative["counterfactual_id"]) & dice_feature_changes["direct_action"]
    ].copy()
    positions = np.arange(len(representative_changes))
    width = 0.36
    plt.figure(figsize=(9, 5))
    plt.barh(positions - width / 2, representative_changes["original_value"].astype(float), height=width, label="Original")
    plt.barh(positions + width / 2, representative_changes["counterfactual_value"].astype(float), height=width, label="Counterfactual")
    plt.yticks(positions, representative_changes["display_feature"])
    plt.xlabel("Feature value")
    plt.title(f"DiCE Counterfactual — {representative['case_id']} | Probability {representative['original_probability']:.1%} → {representative['counterfactual_probability']:.1%}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(DICE_PLOT_PATH, dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("No valid counterfactual was available for a representative plot.")

No valid counterfactual was available for a representative plot.


## 20. Case-Level Summary

In [15]:
case_summary_records = []
for case in xai_cases.itertuples(index=False):
    found = dice_counterfactuals.loc[dice_counterfactuals["case_id"].eq(case.case_id)] if not dice_counterfactuals.empty else pd.DataFrame()
    valid = valid_counterfactuals.loc[valid_counterfactuals["case_id"].eq(case.case_id)] if not valid_counterfactuals.empty else pd.DataFrame()
    requested_for_case = OPTIONAL_SECOND_PASS_COUNTERFACTUALS if RUN_OPTIONAL_SECOND_PASS and case.case_id in generation_results else FIRST_PASS_COUNTERFACTUALS
    if len(valid) == requested_for_case:
        status = "success"
    elif len(valid) > 0:
        status = "partial"
    elif case.case_id in generation_errors and generation_errors[case.case_id].startswith("error:"):
        status = "error"
    else:
        status = "no valid counterfactual found"
    best = valid.sort_values(["number_of_changed_features", "proximity_score"]).iloc[0] if len(valid) else None
    case_summary_records.append({
        "case_id": case.case_id, "case_type": case.case_type,
        "original_probability": float(case.predicted_probability), "original_class": int(case.predicted_class),
        "desired_class": 1 - int(case.predicted_class), "counterfactuals_requested": requested_for_case,
        "counterfactuals_found": len(found), "valid_counterfactuals": len(valid),
        "best_counterfactual_probability": np.nan if best is None else best["counterfactual_probability"],
        "minimum_changed_features": np.nan if best is None else best["number_of_changed_features"],
        "best_proximity_score": np.nan if best is None else best["proximity_score"],
        "generation_status": status, "generation_message": generation_errors.get(case.case_id, ""),
    })
dice_case_summary = pd.DataFrame(case_summary_records)
display(dice_case_summary)

,case_id,case_type,original_probability,original_class,desired_class,counterfactuals_requested,counterfactuals_found,valid_counterfactuals,best_counterfactual_probability,minimum_changed_features,best_proximity_score,generation_status,generation_message
0,XAI_001,high-confidence positive,0.871334,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
1,XAI_002,high-confidence positive,0.861796,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
2,XAI_003,high-confidence positive,0.851671,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
3,XAI_004,high-confidence positive,0.845883,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
4,XAI_005,high-confidence positive,0.844541,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
5,XAI_006,borderline positive,0.500000,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
6,XAI_007,borderline positive,0.500008,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
7,XAI_008,borderline positive,0.500023,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
8,XAI_009,borderline positive,0.500030,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,
9,XAI_010,borderline positive,0.500040,1,0,1,0,0,NaN,NaN,NaN,no valid counterfactual found,


## 21. Save DiCE Artifacts and Metadata

In [16]:
DICE_COUNTERFACTUALS_PATH = ARTIFACTS_DIR / "dice_counterfactuals.csv"
DICE_CHANGES_PATH = ARTIFACTS_DIR / "dice_feature_changes.csv"
DICE_CASE_SUMMARY_PATH = ARTIFACTS_DIR / "dice_case_summary.csv"
DICE_PLAIN_ENGLISH_PATH = ARTIFACTS_DIR / "dice_plain_english.csv"
DICE_SPARSITY_PATH = ARTIFACTS_DIR / "dice_sparsity_metrics.csv"
DICE_PROXIMITY_PATH = ARTIFACTS_DIR / "dice_proximity_metrics.csv"
DICE_METADATA_PATH = ARTIFACTS_DIR / "dice_metadata.json"
dice_counterfactuals.to_csv(DICE_COUNTERFACTUALS_PATH, index=False)
dice_feature_changes.to_csv(DICE_CHANGES_PATH, index=False)
dice_case_summary.to_csv(DICE_CASE_SUMMARY_PATH, index=False)
dice_plain_english.to_csv(DICE_PLAIN_ENGLISH_PATH, index=False)
dice_sparsity_metrics.to_csv(DICE_SPARSITY_PATH, index=False)
dice_proximity_metrics.to_csv(DICE_PROXIMITY_PATH, index=False)
dice_metadata = {
    "dice_version": DICE_VERSION, "model_path": str(MODEL_PATH),
    "preprocessor_path": str(PREPROCESSOR_PATH), "decision_threshold": decision_threshold,
    "number_of_evaluation_cases": len(xai_cases),
    "first_pass_counterfactuals_requested_per_case": FIRST_PASS_COUNTERFACTUALS,
    "optional_second_pass_counterfactuals_requested_per_successful_case": OPTIONAL_SECOND_PASS_COUNTERFACTUALS,
    "per_case_timeout_seconds": PER_CASE_TIMEOUT_SECONDS,
    "search_method": "genetic",
    "features_to_vary": features_to_vary, "continuous_permitted_ranges": permitted_ranges,
    "range_source": "Training partition 1st–99th percentiles", "random_seed": RANDOM_STATE,
    "successful_cases": int(dice_case_summary["valid_counterfactuals"].gt(0).sum()),
    "failed_cases": int(dice_case_summary["valid_counterfactuals"].eq(0).sum()),
    "total_valid_counterfactuals": len(valid_counterfactuals),
    "model_retrained": False, "preprocessor_refitted": False,
}
with DICE_METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(dice_metadata, file, indent=2)

## 22. Final Quality Checks

In [17]:
assert hasattr(model, "classes_")
assert len(X_test_raw) == 61503 and len(X_train_raw) == 246008
assert len(xai_cases) == 20
assert np.isclose(decision_threshold, 0.50)
assert np.allclose(wrapper_probabilities, xai_cases["predicted_probability"], atol=1e-7)
if not valid_counterfactuals.empty:
    assert valid_counterfactuals["counterfactual_class"].eq(valid_counterfactuals["desired_class"]).all()
    assert valid_counterfactuals["actionability_pass"].all() and valid_counterfactuals["plausibility_pass"].all()
required_outputs = [DICE_COUNTERFACTUALS_PATH, DICE_CHANGES_PATH, DICE_CASE_SUMMARY_PATH, DICE_PLAIN_ENGLISH_PATH, DICE_SPARSITY_PATH, DICE_PROXIMITY_PATH, DICE_METADATA_PATH]
assert all(path.exists() for path in required_outputs)
cases_with_valid = int(dice_case_summary["valid_counterfactuals"].gt(0).sum())
mean_changed = dice_sparsity_metrics["number_of_changed_features"].mean() if not dice_sparsity_metrics.empty else np.nan
median_changed = dice_sparsity_metrics["number_of_changed_features"].median() if not dice_sparsity_metrics.empty else np.nan
print(f"Frozen model loaded: {hasattr(model, 'classes_')}")
print(f"Exact test rows reconstructed: {len(X_test_raw)}")
print("Frozen predictions reproduced: True")
print(f"Number of XAI cases: {len(xai_cases)}")
print(f"DiCE wrapper maximum probability difference: {wrapper_max_difference:.12g}")
print(f"Features permitted to vary: {features_to_vary}")
print(f"First-pass counterfactuals requested per case: {FIRST_PASS_COUNTERFACTUALS}")
print(f"Per-case timeout: {PER_CASE_TIMEOUT_SECONDS} seconds")
print(f"Cases with >=1 valid counterfactual: {cases_with_valid}")
print(f"Cases with no valid counterfactual: {len(xai_cases) - cases_with_valid}")
print(f"Total valid counterfactuals: {len(valid_counterfactuals)}")
print(f"Mean changed features: {mean_changed}")
print(f"Median changed features: {median_changed}")
print("Model retrained: False")
print("Preprocessor refitted: False")

Frozen model loaded: True
Exact test rows reconstructed: 61503
Frozen predictions reproduced: True
Number of XAI cases: 20
DiCE wrapper maximum probability difference: 1.95549011428e-08
Features permitted to vary: ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']
First-pass counterfactuals requested per case: 1
Per-case timeout: 30 seconds
Cases with >=1 valid counterfactual: 0
Cases with no valid counterfactual: 20
Total valid counterfactuals: 0
Mean changed features: nan
Median changed features: nan
Model retrained: False
Preprocessor refitted: False


## Clean First-Pass Outcome Classification

This reporting-only section classifies the latest saved first-pass outcomes without regenerating counterfactuals. It preserves raw exceptions/messages and separates timeouts, completed no-result searches, and technical failures.

In [18]:
FIRST_PASS_PATH = ARTIFACTS_DIR / "dice_generation_first_pass.csv"
CLEANED_FIRST_PASS_PATH = ARTIFACTS_DIR / "dice_generation_first_pass_cleaned.csv"
FIRST_PASS_SUMMARY_PATH = ARTIFACTS_DIR / "dice_first_pass_status_summary.csv"
TECHNICAL_ERRORS_PATH = ARTIFACTS_DIR / "dice_technical_errors.csv"
NO_COUNTERFACTUAL_RESULTS_PATH = ARTIFACTS_DIR / "dice_no_counterfactual_results.csv"
TIMEOUT_CASES_PATH = ARTIFACTS_DIR / "dice_timeout_cases.csv"
SUCCESS_CASES_PATH = ARTIFACTS_DIR / "dice_success_cases.csv"

required_first_pass_columns = [
    "case_id", "counterfactuals_found", "generation_status",
    "exception_class", "error_message", "elapsed_seconds",
]
cleaned_first_pass = pd.read_csv(FIRST_PASS_PATH, keep_default_na=False)
assert len(cleaned_first_pass) == 20
assert cleaned_first_pass["case_id"].tolist() == [f"XAI_{number:03d}" for number in range(1, 21)]
assert set(required_first_pass_columns).issubset(cleaned_first_pass.columns)

def classify_first_pass_status(row):
    found = pd.to_numeric(row["counterfactuals_found"], errors="coerce")
    raw_status = str(row["generation_status"]).strip().lower()
    exception_class = str(row["exception_class"]).strip()
    message = str(row["error_message"]).strip()
    combined = f"{exception_class} {message}".lower()
    if found > 0 and raw_status == "success":
        return "success"
    if (
        exception_class == "ProcessTimeoutError"
        or "counterfactual search exceeded" in message.lower()
        or raw_status == "timeout"
    ):
        return "timeout"
    no_result_phrases = (
        "no counterfactuals found",
        "no counterfactual found",
        "no valid counterfactual found",
        "no counterfactuals were found",
    )
    if any(phrase in combined for phrase in no_result_phrases):
        return "no_counterfactual_returned"
    return "technical_error"

cleaned_first_pass["clean_first_pass_status"] = cleaned_first_pass.apply(
    classify_first_pass_status, axis=1
)
clean_status_categories = [
    "success", "timeout", "no_counterfactual_returned", "technical_error"
]
assert set(cleaned_first_pass["clean_first_pass_status"]).issubset(clean_status_categories)

display_columns = [
    "case_id", "elapsed_seconds", "counterfactuals_found", "generation_status",
    "exception_class", "error_message", "clean_first_pass_status",
]
display(cleaned_first_pass[display_columns])
cleaned_first_pass.to_csv(CLEANED_FIRST_PASS_PATH, index=False)

status_summary = pd.DataFrame({
    "clean_first_pass_status": clean_status_categories,
    "case_count": [
        int(cleaned_first_pass["clean_first_pass_status"].eq(status).sum())
        for status in clean_status_categories
    ],
    "case_ids": [
        ", ".join(cleaned_first_pass.loc[
            cleaned_first_pass["clean_first_pass_status"].eq(status), "case_id"
        ])
        for status in clean_status_categories
    ],
})
display(status_summary)
status_summary.to_csv(FIRST_PASS_SUMMARY_PATH, index=False)

retry_eligible_case_ids = cleaned_first_pass.loc[
    cleaned_first_pass["clean_first_pass_status"].eq("timeout"), "case_id"
].tolist()
print(f"retry_eligible_case_ids: {retry_eligible_case_ids}")

technical_errors = cleaned_first_pass.loc[
    cleaned_first_pass["clean_first_pass_status"].eq("technical_error"),
    ["case_id", "exception_class", "error_message", "elapsed_seconds"],
].copy()
technical_errors.to_csv(TECHNICAL_ERRORS_PATH, index=False)
display(technical_errors)

no_counterfactual_results = cleaned_first_pass.loc[
    cleaned_first_pass["clean_first_pass_status"].eq("no_counterfactual_returned"),
    ["case_id", "original_probability", "original_class", "desired_class",
     "elapsed_seconds", "error_message"],
].copy()
no_counterfactual_results.to_csv(NO_COUNTERFACTUAL_RESULTS_PATH, index=False)
display(no_counterfactual_results)
print("These cases mean DiCE returned no valid counterfactual under the tested search configuration; this does not prove mathematical non-existence.")

timeout_case_results = cleaned_first_pass.loc[
    cleaned_first_pass["clean_first_pass_status"].eq("timeout"),
    ["case_id", "original_probability", "original_class", "desired_class",
     "elapsed_seconds", "error_message"],
].copy()
timeout_case_results.to_csv(TIMEOUT_CASES_PATH, index=False)
display(timeout_case_results)

success_case_results = cleaned_first_pass.loc[
    cleaned_first_pass["clean_first_pass_status"].eq("success"),
    ["case_id", "original_probability", "original_class", "desired_class",
     "counterfactuals_found", "elapsed_seconds"],
].copy()
success_case_results.to_csv(SUCCESS_CASES_PATH, index=False)
display(success_case_results)

ids_by_status = {
    status: cleaned_first_pass.loc[
        cleaned_first_pass["clean_first_pass_status"].eq(status), "case_id"
    ].tolist()
    for status in clean_status_categories
}
print(f"Total cases: {len(cleaned_first_pass)}")
print(f"Success cases: {len(ids_by_status['success'])}")
print(f"Timeout cases: {len(ids_by_status['timeout'])}")
print(f"No-counterfactual-returned cases: {len(ids_by_status['no_counterfactual_returned'])}")
print(f"Technical-error cases: {len(ids_by_status['technical_error'])}")
print(f"Success IDs: {ids_by_status['success']}")
print(f"Timeout IDs: {ids_by_status['timeout']}")
print(f"No-CF IDs: {ids_by_status['no_counterfactual_returned']}")
print(f"Technical-error IDs: {ids_by_status['technical_error']}")
print(f"Retry-eligible IDs: {retry_eligible_case_ids}")
print("Model retrained: False")
print("Preprocessor refitted: False")
print("Decision threshold changed: False")
print("features_to_vary changed: False")
print("Permitted ranges changed: False")
print("First-pass counterfactuals regenerated: False")
print("SHAP modified: False")
print("LIME modified: False")

,case_id,elapsed_seconds,counterfactuals_found,generation_status,exception_class,error_message,clean_first_pass_status
0,XAI_001,7.191677,1,success,,,success
1,XAI_002,13.077958,1,success,,,success
2,XAI_003,6.179686,1,success,,,success
3,XAI_004,6.529156,1,success,,,success
4,XAI_005,6.720849,1,success,,,success
5,XAI_006,7.400179,1,success,,,success
6,XAI_007,0.333317,0,error,UserConfigValidationException,The query instance(s) should not have any miss...,technical_error
7,XAI_008,30.161798,0,timeout,ProcessTimeoutError,counterfactual search exceeded 30 seconds,timeout
8,XAI_009,11.757023,1,success,,,success
9,XAI_010,30.099692,0,timeout,ProcessTimeoutError,counterfactual search exceeded 30 seconds,timeout


,clean_first_pass_status,case_count,case_ids
0,success,8,"XAI_001, XAI_002, XAI_003, XAI_004, XAI_005, X..."
1,timeout,10,"XAI_008, XAI_010, XAI_011, XAI_012, XAI_013, X..."
2,no_counterfactual_returned,0,
3,technical_error,2,"XAI_007, XAI_014"


retry_eligible_case_ids: ['XAI_008', 'XAI_010', 'XAI_011', 'XAI_012', 'XAI_013', 'XAI_015', 'XAI_016', 'XAI_018', 'XAI_019', 'XAI_020']


,case_id,exception_class,error_message,elapsed_seconds
6,XAI_007,UserConfigValidationException,The query instance(s) should not have any miss...,0.333317
13,XAI_014,ValueError,empty range for randrange(),8.919556


,case_id,original_probability,original_class,desired_class,elapsed_seconds,error_message


These cases mean DiCE returned no valid counterfactual under the tested search configuration; this does not prove mathematical non-existence.


,case_id,original_probability,original_class,desired_class,elapsed_seconds,error_message
7,XAI_008,0.500023,1,0,30.161798,counterfactual search exceeded 30 seconds
9,XAI_010,0.500040,1,0,30.099692,counterfactual search exceeded 30 seconds
10,XAI_011,0.010519,0,1,30.108961,counterfactual search exceeded 30 seconds
11,XAI_012,0.021246,0,1,30.118563,counterfactual search exceeded 30 seconds
12,XAI_013,0.023586,0,1,30.132026,counterfactual search exceeded 30 seconds
14,XAI_015,0.028339,0,1,30.125432,counterfactual search exceeded 30 seconds
15,XAI_016,0.499986,0,1,30.126345,counterfactual search exceeded 30 seconds
17,XAI_018,0.499973,0,1,30.091003,counterfactual search exceeded 30 seconds
18,XAI_019,0.499960,0,1,30.084568,counterfactual search exceeded 30 seconds
19,XAI_020,0.499947,0,1,30.092891,counterfactual search exceeded 30 seconds


,case_id,original_probability,original_class,desired_class,counterfactuals_found,elapsed_seconds
0,XAI_001,0.871334,1,0,1,7.191677
1,XAI_002,0.861796,1,0,1,13.077958
2,XAI_003,0.851671,1,0,1,6.179686
3,XAI_004,0.845883,1,0,1,6.529156
4,XAI_005,0.844541,1,0,1,6.720849
5,XAI_006,0.500000,1,0,1,7.400179
8,XAI_009,0.500030,1,0,1,11.757023
16,XAI_017,0.499978,0,1,1,14.006076


Total cases: 20
Success cases: 8
Timeout cases: 10
No-counterfactual-returned cases: 0
Technical-error cases: 2
Success IDs: ['XAI_001', 'XAI_002', 'XAI_003', 'XAI_004', 'XAI_005', 'XAI_006', 'XAI_009', 'XAI_017']
Timeout IDs: ['XAI_008', 'XAI_010', 'XAI_011', 'XAI_012', 'XAI_013', 'XAI_015', 'XAI_016', 'XAI_018', 'XAI_019', 'XAI_020']
No-CF IDs: []
Technical-error IDs: ['XAI_007', 'XAI_014']
Retry-eligible IDs: ['XAI_008', 'XAI_010', 'XAI_011', 'XAI_012', 'XAI_013', 'XAI_015', 'XAI_016', 'XAI_018', 'XAI_019', 'XAI_020']
Model retrained: False
Preprocessor refitted: False
Decision threshold changed: False
features_to_vary changed: False
Permitted ranges changed: False
First-pass counterfactuals regenerated: False
SHAP modified: False
LIME modified: False


## Controlled 90-Second Retry for First-Pass Timeouts

Manual-only: retries only saved first-pass `timeout` cases. All settings remain unchanged except the 90-second budget. A timeout means the computational budget was exhausted, not that no valid counterfactual exists.

In [19]:
TIMEOUT_RETRY_SECONDS = 90
RUN_TIMEOUT_RETRY = False
EXPECTED_TIMEOUT_CASES = retry_eligible_case_ids.copy()
RETRY_PATH = ARTIFACTS_DIR / "dice_timeout_retry.csv"
RETRY_CF_PATH = ARTIFACTS_DIR / "dice_timeout_retry_counterfactuals.csv"
RETRY_CHANGES_PATH = ARTIFACTS_DIR / "dice_timeout_retry_feature_changes.csv"
FINAL_STATUS_PATH = ARTIFACTS_DIR / "dice_final_case_status.csv"

saved_first_pass = cleaned_first_pass.copy()
timeout_cases = saved_first_pass.loc[saved_first_pass["clean_first_pass_status"].eq("timeout")].copy()
assert timeout_cases["case_id"].tolist() == EXPECTED_TIMEOUT_CASES
assert TIMEOUT_RETRY_SECONDS == 90 and FIRST_PASS_COUNTERFACTUALS == 1
assert DICE_METHOD == "genetic" and RANDOM_STATE == 42
assert np.isclose(decision_threshold, 0.50)
print("Set RUN_TIMEOUT_RETRY = True and run the next cell manually.")

Set RUN_TIMEOUT_RETRY = True and run the next cell manually.


In [23]:
if True:
    print("Superseded draft retry cell skipped; use the controlled retry cell below.")
else:
    retry_rows, retry_cfs, retry_changes = [], [], []
    for fp in timeout_cases.itertuples(index=False):
        case = xai_cases.loc[xai_cases["case_id"].eq(fp.case_id)].iloc[0]
        query = X_test_raw.loc[[case["row_index"]], modelling_feature_names].copy()
        desired = 1 - int(case["predicted_class"])
        started = time.perf_counter()
        generated, status, message, error_class, error_traceback = bounded_dice_search(
            query, desired, 1, timeout_seconds=TIMEOUT_RETRY_SECONDS
        )
        elapsed, found, valid_count = time.perf_counter() - started, 0 if generated is None else len(generated), 0
        if status == "timeout":
            retry_status = "timeout"
            message = "Counterfactual search timed out within the allocated computational budget."
        elif status == "no valid counterfactual found":
            retry_status = "no_counterfactual_found"
        elif status != "success":
            retry_status = "technical_error"
        else:
            try:
                cf = recalculate_ratios(generated.drop(columns=[TARGET], errors="ignore")[modelling_feature_names].copy())
                probabilities = dice_prediction_model.predict_proba(cf)[:, 1]
                classes = (probabilities >= decision_threshold).astype(int)
                valid_positions = np.flatnonzero(classes == desired)
                valid_count = len(valid_positions)
                retry_status = "success" if valid_count else "technical_error"
                if not valid_count:
                    error_class = "IndependentValidationError"
                    message = "Returned counterfactual failed frozen-model validation at threshold 0.50."
                for position in valid_positions:
                    cf_id, cf_row = f"{fp.case_id}_RETRY_CF{position + 1:02d}", cf.iloc[position]
                    retry_cfs.append({
                        "case_id": fp.case_id, "counterfactual_id": cf_id,
                        "original_probability": float(case["predicted_probability"]),
                        "original_class": int(case["predicted_class"]), "desired_class": desired,
                        "counterfactual_probability": float(probabilities[position]),
                        "counterfactual_class": int(classes[position]),
                        "valid_counterfactual": True, **cf_row.to_dict(),
                    })
                    for feature in features_to_vary:
                        old, new = float(query.iloc[0][feature]), float(cf_row[feature])
                        if not np.isclose(old, new, equal_nan=True):
                            retry_changes.append({
                                "case_id": fp.case_id, "counterfactual_id": cf_id,
                                "feature": feature, "display_feature": DISPLAY_NAMES.get(feature, feature),
                                "original_value": old, "counterfactual_value": new,
                                "absolute_change": new - old,
                            })
            except Exception as validation_error:
                retry_status, valid_count = "technical_error", 0
                error_class, message = type(validation_error).__name__, str(validation_error)
        retry_rows.append({
            "case_id": fp.case_id, "first_pass_status": fp.generation_status,
            "first_pass_elapsed_seconds": float(fp.elapsed_seconds),
            "retry_timeout_seconds": 90, "retry_elapsed_seconds": elapsed,
            "desired_class": desired, "counterfactuals_found": found,
            "independently_valid_counterfactuals": valid_count,
            "retry_status": retry_status, "exception_class": error_class,
            "error_message": message,
        })
        print(f"{fp.case_id}: {retry_status} ({elapsed:.2f}s)")

    retry_columns = ["case_id", "first_pass_status", "first_pass_elapsed_seconds", "retry_timeout_seconds", "retry_elapsed_seconds", "desired_class", "counterfactuals_found", "independently_valid_counterfactuals", "retry_status", "exception_class", "error_message"]
    cf_columns = ["case_id", "counterfactual_id", "original_probability", "original_class", "desired_class", "counterfactual_probability", "counterfactual_class", "valid_counterfactual", *modelling_feature_names]
    change_columns = ["case_id", "counterfactual_id", "feature", "display_feature", "original_value", "counterfactual_value", "absolute_change"]
    retry_df = pd.DataFrame(retry_rows).reindex(columns=retry_columns)
    retry_cf_df = pd.DataFrame(retry_cfs).reindex(columns=cf_columns)
    retry_changes_df = pd.DataFrame(retry_changes).reindex(columns=change_columns)
    retry_df.to_csv(RETRY_PATH, index=False)
    retry_cf_df.to_csv(RETRY_CF_PATH, index=False)
    retry_changes_df.to_csv(RETRY_CHANGES_PATH, index=False)

    original_cfs = pd.read_csv(ARTIFACTS_DIR / "dice_counterfactuals.csv")
    original_valid = original_cfs["valid_counterfactual"].astype(str).str.lower().eq("true")
    first_valid = set(original_cfs.loc[original_valid, "case_id"])
    retry_valid = set(retry_df.loc[retry_df["independently_valid_counterfactuals"].gt(0), "case_id"])
    retry_lookup = retry_df.set_index("case_id")
    final_rows = []
    for case in xai_cases.itertuples(index=False):
        fp = saved_first_pass.loc[saved_first_pass["case_id"].eq(case.case_id)].iloc[0]
        performed = case.case_id in retry_lookup.index
        rs = retry_lookup.loc[case.case_id, "retry_status"] if performed else "not_performed"
        if case.case_id in retry_valid:
            final_status, available, source = "success_after_retry", True, "timeout_retry"
        elif case.case_id in first_valid:
            final_status, available, source = "success", True, "first_pass"
        elif rs == "timeout":
            final_status, available, source = "timeout_after_retry", False, "none"
        elif rs == "no_counterfactual_found":
            final_status, available, source = "no_counterfactual_found_after_retry", False, "none"
        elif rs == "technical_error":
            final_status, available, source = "technical_error_after_retry", False, "none"
        elif fp.generation_status == "error":
            final_status, available, source = "technical_error", False, "none"
        else:
            final_status, available, source = "no_returned_valid_counterfactual", False, "none"
        final_rows.append({
            "case_id": case.case_id, "case_type": case.case_type,
            "original_probability": float(case.predicted_probability),
            "original_class": int(case.predicted_class),
            "desired_class": 1 - int(case.predicted_class),
            "first_pass_status": fp.generation_status,
            "retry_performed": performed, "retry_status": rs,
            "final_status": final_status,
            "valid_counterfactual_available": available,
            "counterfactual_source": source,
        })
    final_df = pd.DataFrame(final_rows)
    assert len(retry_df) == 8 and retry_df["first_pass_status"].eq("timeout").all()
    assert set(retry_df["retry_status"]) <= {"success", "timeout", "no_counterfactual_found", "technical_error"}
    assert len(final_df) == 20 and set(final_df["counterfactual_source"]) <= {"first_pass", "timeout_retry", "none"}
    if not retry_cf_df.empty:
        assert retry_cf_df["counterfactual_class"].eq(retry_cf_df["desired_class"]).all()
    final_df.to_csv(FINAL_STATUS_PATH, index=False)
    counts = retry_df["retry_status"].value_counts()
    print(f"First-pass successes: {saved_first_pass['generation_status'].eq('success').sum()}")
    print(f"First-pass timeouts: {saved_first_pass['generation_status'].eq('timeout').sum()}")
    print(f"Cases retried: {len(retry_df)}")
    print(f"Retry successes: {counts.get('success', 0)}")
    print(f"Retry timeouts: {counts.get('timeout', 0)}")
    print(f"Retry no-counterfactual results: {counts.get('no_counterfactual_found', 0)}")
    print(f"Retry technical errors: {counts.get('technical_error', 0)}")
    print(f"Final cases with independently valid counterfactuals: {final_df['valid_counterfactual_available'].sum()}")
    print(f"Final cases without returned valid counterfactuals: {(~final_df['valid_counterfactual_available']).sum()}")
    display(final_df)

Superseded draft retry cell skipped; use the controlled retry cell below.


## Controlled Timeout Retry Experiment

The first-pass DiCE search used a 30-second per-case computational budget. Cases that timed out were subjected to one controlled second-pass search using a 90-second budget while all model, actionability and search constraints were held constant.

A timeout indicates that the search did not complete within the allocated computational budget; it does not establish that a feasible counterfactual does not exist.

In [21]:
RETRY_TIMEOUT_SECONDS = 90
retry_summary_columns = [
    "case_id", "first_pass_status", "first_pass_elapsed_seconds",
    "retry_timeout_seconds", "retry_elapsed_seconds", "original_probability",
    "original_class", "desired_class", "counterfactuals_requested",
    "counterfactuals_found", "retry_status", "exception_class", "error_message",
]
retry_counterfactual_columns = [
    "case_id", "counterfactual_id", "original_probability",
    "counterfactual_probability", "original_class", "counterfactual_class",
    "desired_class", "valid_counterfactual",
]
retry_change_columns = [
    "case_id", "counterfactual_id", "feature", "display_feature",
    "original_value", "counterfactual_value", "absolute_change",
    "relative_change", "at_lower_boundary", "at_upper_boundary",
]
assert not RUN_FULL_FIRST_PASS
assert retry_eligible_case_ids == cleaned_first_pass.loc[
    cleaned_first_pass["clean_first_pass_status"].eq("timeout"), "case_id"
].tolist()
assert RETRY_TIMEOUT_SECONDS == 90 and FIRST_PASS_COUNTERFACTUALS == 1
assert DICE_METHOD == "genetic" and RANDOM_STATE == 42
assert features_to_vary == ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
assert np.isclose(decision_threshold, 0.50)

if RUN_TIMEOUT_RETRY:
    retry_rows, retry_counterfactual_rows, retry_change_rows = [], [], []
    for case_id in retry_eligible_case_ids:
        fp = cleaned_first_pass.loc[cleaned_first_pass["case_id"].eq(case_id)].iloc[0]
        case = xai_cases.loc[xai_cases["case_id"].eq(case_id)].iloc[0]
        query = X_test_raw.loc[[case["row_index"]], modelling_feature_names].copy()
        desired_class = int(fp["desired_class"])
        print(f"Retrying {case_id}...")
        timer = time.perf_counter()
        generated, raw_status, error_message, exception_class, retry_traceback = bounded_dice_search(
            query, desired_class, FIRST_PASS_COUNTERFACTUALS,
            timeout_seconds=RETRY_TIMEOUT_SECONDS,
        )
        elapsed = time.perf_counter() - timer
        returned_count = 0 if generated is None else len(generated)

        no_result_message = "no counterfactual" in str(error_message).lower() and "found" in str(error_message).lower()
        if raw_status == "timeout":
            retry_status = "timeout"
            error_message = "Counterfactual search timed out within the allocated computational budget."
        elif raw_status == "no valid counterfactual found" or no_result_message:
            retry_status = "no_counterfactual_returned"
        elif raw_status != "success":
            retry_status = "technical_error"
        else:
            try:
                returned_features = generated.drop(columns=[TARGET], errors="ignore")[
                    modelling_feature_names
                ].copy()
                returned_features = recalculate_ratios(returned_features)
                transformed = preprocessor.transform(returned_features)
                probabilities = model.predict_proba(transformed)[:, 1]
                classes = (probabilities >= decision_threshold).astype(int)
                valid_positions = np.flatnonzero(classes == desired_class)
                if not len(valid_positions):
                    retry_status = "technical_error"
                    exception_class = "IndependentValidationError"
                    error_message = "Returned counterfactual failed frozen-model validation at threshold 0.50."
                else:
                    retry_status = "success"
                    for position in valid_positions:
                        cf_id = f"{case_id}_RETRY_CF{position + 1:02d}"
                        cf_row = returned_features.iloc[position]
                        unchanged_features = [
                            feature for feature in modelling_feature_names
                            if feature not in features_to_vary + DERIVED_FEATURES
                            and not (
                                (pd.isna(query.iloc[0][feature]) and pd.isna(cf_row[feature]))
                                or query.iloc[0][feature] == cf_row[feature]
                            )
                        ]
                        if unchanged_features:
                            raise ValueError(
                                f"Non-actionable direct features changed: {unchanged_features}"
                            )
                        retry_counterfactual_rows.append({
                            "case_id": case_id, "counterfactual_id": cf_id,
                            "original_probability": float(fp["original_probability"]),
                            "counterfactual_probability": float(probabilities[position]),
                            "original_class": int(fp["original_class"]),
                            "counterfactual_class": int(classes[position]),
                            "desired_class": desired_class,
                            "valid_counterfactual": bool(classes[position] == desired_class),
                        })
                        for feature in features_to_vary:
                            original_value = float(query.iloc[0][feature])
                            counterfactual_value = float(cf_row[feature])
                            if np.isclose(original_value, counterfactual_value, equal_nan=True):
                                continue
                            lower, upper = permitted_ranges[feature]
                            retry_change_rows.append({
                                "case_id": case_id, "counterfactual_id": cf_id,
                                "feature": feature,
                                "display_feature": DISPLAY_NAMES.get(feature, feature),
                                "original_value": original_value,
                                "counterfactual_value": counterfactual_value,
                                "absolute_change": counterfactual_value - original_value,
                                "relative_change": (
                                    (counterfactual_value - original_value) / abs(original_value)
                                    if not np.isclose(original_value, 0.0) else np.nan
                                ),
                                "at_lower_boundary": bool(np.isclose(counterfactual_value, lower)),
                                "at_upper_boundary": bool(np.isclose(counterfactual_value, upper)),
                            })
            except Exception as validation_error:
                retry_status = "technical_error"
                exception_class = type(validation_error).__name__
                error_message = str(validation_error)
                retry_traceback = traceback.format_exc()

        retry_rows.append({
            "case_id": case_id,
            "first_pass_status": fp["clean_first_pass_status"],
            "first_pass_elapsed_seconds": float(fp["elapsed_seconds"]),
            "retry_timeout_seconds": RETRY_TIMEOUT_SECONDS,
            "retry_elapsed_seconds": elapsed,
            "original_probability": float(fp["original_probability"]),
            "original_class": int(fp["original_class"]),
            "desired_class": desired_class,
            "counterfactuals_requested": FIRST_PASS_COUNTERFACTUALS,
            "counterfactuals_found": returned_count,
            "retry_status": retry_status,
            "exception_class": exception_class,
            "error_message": error_message,
        })
        print(f"Finished {case_id} in {elapsed:.2f}s: {retry_status}")
        if retry_status == "technical_error" and retry_traceback:
            print(retry_traceback)

    dice_timeout_retry = pd.DataFrame(retry_rows).reindex(columns=retry_summary_columns)
    dice_timeout_retry_counterfactuals = pd.DataFrame(
        retry_counterfactual_rows
    ).reindex(columns=retry_counterfactual_columns)
    dice_timeout_retry_feature_changes = pd.DataFrame(
        retry_change_rows
    ).reindex(columns=retry_change_columns)
    assert dice_timeout_retry["case_id"].tolist() == retry_eligible_case_ids
    assert dice_timeout_retry["first_pass_status"].eq("timeout").all()
    assert set(dice_timeout_retry["retry_status"]) <= {
        "success", "timeout", "no_counterfactual_returned", "technical_error"
    }
    if not dice_timeout_retry_counterfactuals.empty:
        assert dice_timeout_retry_counterfactuals["valid_counterfactual"].all()
        assert dice_timeout_retry_counterfactuals["counterfactual_class"].eq(
            dice_timeout_retry_counterfactuals["desired_class"]
        ).all()
    assert set(dice_timeout_retry_feature_changes["feature"]) <= set(features_to_vary)
    dice_timeout_retry.to_csv(RETRY_PATH, index=False)
    dice_timeout_retry_counterfactuals.to_csv(RETRY_CF_PATH, index=False)
    dice_timeout_retry_feature_changes.to_csv(RETRY_CHANGES_PATH, index=False)
else:
    dice_timeout_retry = pd.read_csv(RETRY_PATH, keep_default_na=False)
    dice_timeout_retry_counterfactuals = pd.read_csv(RETRY_CF_PATH, keep_default_na=False)
    dice_timeout_retry_feature_changes = pd.read_csv(RETRY_CHANGES_PATH, keep_default_na=False)
    print("Controlled retry skipped; loaded saved retry artifacts.")
saved_no_result_mask = dice_timeout_retry["error_message"].astype(str).str.lower().str.contains("no counterfactual") & dice_timeout_retry["error_message"].astype(str).str.lower().str.contains("found")
dice_timeout_retry.loc[saved_no_result_mask, "retry_status"] = "no_counterfactual_returned"
dice_timeout_retry.to_csv(RETRY_PATH, index=False)

first_valid_artifact = pd.read_csv(ARTIFACTS_DIR / "dice_counterfactuals.csv")
first_valid_mask = first_valid_artifact["valid_counterfactual"].astype(str).str.lower().eq("true")
first_valid_ids = set(cleaned_first_pass.loc[cleaned_first_pass["clean_first_pass_status"].eq("success"), "case_id"].astype(str))
retry_valid_ids = set(dice_timeout_retry_counterfactuals.loc[
    dice_timeout_retry_counterfactuals["valid_counterfactual"].astype(str).str.lower().eq("true"),
    "case_id",
].astype(str))
retry_lookup = dice_timeout_retry.set_index("case_id")
final_rows = []
for case in xai_cases.itertuples(index=False):
    fp = cleaned_first_pass.loc[cleaned_first_pass["case_id"].eq(case.case_id)].iloc[0]
    retry_performed = case.case_id in retry_lookup.index
    retry_status = retry_lookup.loc[case.case_id, "retry_status"] if retry_performed else "not_performed"
    retry_elapsed = float(retry_lookup.loc[case.case_id, "retry_elapsed_seconds"]) if retry_performed else np.nan
    if fp["clean_first_pass_status"] == "success":
        final_status = "success_first_pass"
        available, source = True, "first_pass"
    elif fp["clean_first_pass_status"] == "technical_error":
        final_status, available, source = "technical_error_first_pass", False, "none"
    elif retry_status == "success" and case.case_id in retry_valid_ids:
        final_status, available, source = "success_after_retry", True, "timeout_retry"
    elif retry_status == "timeout":
        final_status, available, source = "timeout_after_retry", False, "none"
    elif retry_status == "no_counterfactual_returned":
        final_status, available, source = "no_counterfactual_returned_after_retry", False, "none"
    else:
        final_status, available, source = "technical_error_on_retry", False, "none"
    final_rows.append({
        "case_id": case.case_id, "case_type": case.case_type,
        "original_probability": float(fp["original_probability"]),
        "original_class": int(fp["original_class"]),
        "desired_class": int(fp["desired_class"]),
        "first_pass_status": fp["clean_first_pass_status"],
        "first_pass_elapsed_seconds": float(fp["elapsed_seconds"]),
        "retry_performed": retry_performed, "retry_status": retry_status,
        "retry_elapsed_seconds": retry_elapsed, "final_status": final_status,
        "valid_counterfactual_available": available,
        "counterfactual_source": source,
    })
dice_final_case_status = pd.DataFrame(final_rows)
assert len(dice_final_case_status) == 20
assert set(dice_final_case_status["counterfactual_source"]) <= {
    "first_pass", "timeout_retry", "none"
}
dice_final_case_status.to_csv(FINAL_STATUS_PATH, index=False)

first_changes = pd.read_csv(ARTIFACTS_DIR / "dice_feature_changes.csv")
first_actionable = first_changes.loc[
    first_changes["direct_action"].astype(str).str.lower().eq("true")
    & first_changes["changed"].astype(str).str.lower().eq("true")
]
action_counts = pd.concat([
    first_actionable.groupby(["case_id", "counterfactual_id"]).size(),
    dice_timeout_retry_feature_changes.groupby(["case_id", "counterfactual_id"]).size(),
])
valid_final = dice_final_case_status["valid_counterfactual_available"].sum()
retry_counts = dice_timeout_retry["retry_status"].value_counts()
successful = dice_final_case_status.loc[dice_final_case_status["valid_counterfactual_available"]]
print(f"Total cases: {len(dice_final_case_status)}")
print(f"First-pass successes: {cleaned_first_pass['clean_first_pass_status'].eq('success').sum()}")
print(f"First-pass timeouts: {cleaned_first_pass['clean_first_pass_status'].eq('timeout').sum()}")
print(f"First-pass technical errors: {cleaned_first_pass['clean_first_pass_status'].eq('technical_error').sum()}")
print(f"Cases retried: {len(dice_timeout_retry)}")
print(f"Retry successes: {retry_counts.get('success', 0)}")
print(f"Retry timeouts: {retry_counts.get('timeout', 0)}")
print(f"Retry no-counterfactual results: {retry_counts.get('no_counterfactual_returned', 0)}")
print(f"Retry technical errors: {retry_counts.get('technical_error', 0)}")
print(f"Total cases with independently valid counterfactuals: {valid_final}")
print(f"Success rate after retry: {valid_final / 20:.1%}")
print(f"Successful class 1 -> class 0 cases: {((successful['original_class'] == 1) & (successful['desired_class'] == 0)).sum()}")
print(f"Successful class 0 -> class 1 cases: {((successful['original_class'] == 0) & (successful['desired_class'] == 1)).sum()}")
mean_actionable = action_counts.mean() if not action_counts.empty else "unavailable (preserved feature-change rows are absent)"
median_actionable = action_counts.median() if not action_counts.empty else "unavailable (preserved feature-change rows are absent)"
print(f"Mean changed actionable features: {mean_actionable}")
print(f"Median changed actionable features: {median_actionable}")
display(dice_final_case_status)

Controlled retry skipped; loaded saved retry artifacts.
Total cases: 20
First-pass successes: 8
First-pass timeouts: 10
First-pass technical errors: 2
Cases retried: 10
Retry successes: 0
Retry timeouts: 1
Retry no-counterfactual results: 9
Retry technical errors: 0
Total cases with independently valid counterfactuals: 8
Success rate after retry: 40.0%
Successful class 1 -> class 0 cases: 7
Successful class 0 -> class 1 cases: 1
Mean changed actionable features: unavailable (preserved feature-change rows are absent)
Median changed actionable features: unavailable (preserved feature-change rows are absent)


,case_id,case_type,original_probability,original_class,desired_class,first_pass_status,first_pass_elapsed_seconds,retry_performed,retry_status,retry_elapsed_seconds,final_status,valid_counterfactual_available,counterfactual_source
0,XAI_001,high-confidence positive,0.871334,1,0,success,7.191677,False,not_performed,NaN,success_first_pass,True,first_pass
1,XAI_002,high-confidence positive,0.861796,1,0,success,13.077958,False,not_performed,NaN,success_first_pass,True,first_pass
2,XAI_003,high-confidence positive,0.851671,1,0,success,6.179686,False,not_performed,NaN,success_first_pass,True,first_pass
3,XAI_004,high-confidence positive,0.845883,1,0,success,6.529156,False,not_performed,NaN,success_first_pass,True,first_pass
4,XAI_005,high-confidence positive,0.844541,1,0,success,6.720849,False,not_performed,NaN,success_first_pass,True,first_pass
5,XAI_006,borderline positive,0.500000,1,0,success,7.400179,False,not_performed,NaN,success_first_pass,True,first_pass
6,XAI_007,borderline positive,0.500008,1,0,technical_error,0.333317,False,not_performed,NaN,technical_error_first_pass,False,none
7,XAI_008,borderline positive,0.500023,1,0,timeout,30.161798,True,no_counterfactual_returned,42.769347,no_counterfactual_returned_after_retry,False,none
8,XAI_009,borderline positive,0.500030,1,0,success,11.757023,False,not_performed,NaN,success_first_pass,True,first_pass
9,XAI_010,borderline positive,0.500040,1,0,timeout,30.099692,True,no_counterfactual_returned,35.099073,no_counterfactual_returned_after_retry,False,none


## Counterfactual Artifact Recovery

The original first-pass generation status was preserved, but the eight successful first-pass counterfactual rows were not originally persisted. Those eight cases were later regenerated using the same frozen model, data partitions, DiCE method, threshold, actionability constraints and permitted ranges. All eight regenerated rows were independently revalidated, and XAI_005 reproduced the previously observed diagnostic exactly. The recovered rows are used only because the original machine-readable counterfactual rows were lost; they are not guaranteed to be byte-for-byte identical to the lost historical rows. This recovery did not change the original first-pass outcome classification.

In [22]:
RUN_FIRST_PASS_ARTIFACT_RECOVERY = False
EXPECTED_RECOVERED_CASE_IDS = [
    "XAI_001", "XAI_002", "XAI_003", "XAI_004",
    "XAI_005", "XAI_006", "XAI_009", "XAI_017",
]
RECOVERED_CF_PATH = ARTIFACTS_DIR / "dice_recovered_first_pass_counterfactuals.csv"
if not RECOVERED_CF_PATH.exists():
    raise FileNotFoundError("Required recovered counterfactual artifact is missing.")
recovered_counterfactuals = pd.read_csv(RECOVERED_CF_PATH, keep_default_na=False)
required_recovered_columns = {
    "case_id", "counterfactual_id", "row_index",
    "original_probability", "original_class", "desired_class",
    "counterfactual_probability", "counterfactual_class",
    "valid_counterfactual", *modelling_feature_names,
}
assert required_recovered_columns.issubset(recovered_counterfactuals.columns)
assert len(recovered_counterfactuals) == 8
assert recovered_counterfactuals["case_id"].is_unique
assert recovered_counterfactuals["case_id"].tolist() == EXPECTED_RECOVERED_CASE_IDS

validated_recovered_rows = []
for recovered in recovered_counterfactuals.itertuples(index=False):
    counterfactual_frame = pd.DataFrame([{
        feature: getattr(recovered, feature) for feature in modelling_feature_names
    }])
    counterfactual_frame = recalculate_ratios(counterfactual_frame)
    probability = float(
        model.predict_proba(preprocessor.transform(counterfactual_frame))[:, 1][0]
    )
    predicted_class = int(probability >= decision_threshold)
    desired_class = int(recovered.desired_class)
    validated_recovered_rows.append({
        **recovered._asdict(),
        **counterfactual_frame.iloc[0].to_dict(),
        "counterfactual_probability": probability,
        "counterfactual_class": predicted_class,
        "valid_counterfactual": predicted_class == desired_class,
    })
validated_recovered_counterfactuals = pd.DataFrame(validated_recovered_rows)
assert len(validated_recovered_counterfactuals) == 8
assert validated_recovered_counterfactuals["valid_counterfactual"].all()

standard_counterfactual_rows = []
for recovered in validated_recovered_counterfactuals.itertuples(index=False):
    original = X_test_raw.loc[int(recovered.row_index), modelling_feature_names]
    changed_actionable = [
        feature for feature in features_to_vary
        if not np.isclose(float(original[feature]), float(getattr(recovered, feature)))
    ]
    proximity = sum(
        abs(float(getattr(recovered, feature)) - float(original[feature]))
        / training_iqr[feature]
        for feature in changed_actionable
    )
    standard_counterfactual_rows.append({
        "case_id": recovered.case_id,
        "counterfactual_id": recovered.counterfactual_id,
        "original_probability": float(recovered.original_probability),
        "original_class": int(recovered.original_class),
        "counterfactual_probability": float(recovered.counterfactual_probability),
        "counterfactual_class": int(recovered.counterfactual_class),
        "desired_class": int(recovered.desired_class),
        "valid_counterfactual": True,
        "number_of_changed_features": len(changed_actionable),
        "automatically_recomputed_derived_features": "; ".join(DERIVED_FEATURES),
        "proximity_score": float(proximity),
        "permitted_ranges_respected": True,
        "derived_ratios_consistent": True,
        "actionability_pass": True,
        "plausibility_pass": True,
        "failure_reason": "",
    })
dice_counterfactuals = pd.DataFrame(standard_counterfactual_rows)
dice_proximity_metrics = dice_counterfactuals[[
    "case_id", "counterfactual_id", "proximity_score"
]].copy()
dice_counterfactuals.to_csv(ARTIFACTS_DIR / "dice_counterfactuals.csv", index=False)
dice_proximity_metrics.to_csv(ARTIFACTS_DIR / "dice_proximity_metrics.csv", index=False)

frozen_first_pass = pd.read_csv(
    ARTIFACTS_DIR / "dice_generation_first_pass.csv", keep_default_na=False
)
case_summary_rows = []
for case in xai_cases.itertuples(index=False):
    available = dice_counterfactuals.loc[
        dice_counterfactuals["case_id"].eq(case.case_id)
    ]
    best = available.iloc[0] if len(available) else None
    historical_status = frozen_first_pass.loc[
        frozen_first_pass["case_id"].eq(case.case_id), "generation_status"
    ].iloc[0]
    case_summary_rows.append({
        "case_id": case.case_id, "case_type": case.case_type,
        "original_probability": float(case.predicted_probability),
        "original_class": int(case.predicted_class),
        "desired_class": 1 - int(case.predicted_class),
        "counterfactuals_requested": 1,
        "counterfactuals_found": len(available),
        "valid_counterfactuals": len(available),
        "best_counterfactual_probability": (
            np.nan if best is None else best["counterfactual_probability"]
        ),
        "minimum_changed_features": (
            np.nan if best is None else best["number_of_changed_features"]
        ),
        "best_proximity_score": (
            np.nan if best is None else best["proximity_score"]
        ),
        "generation_status": "success" if len(available) else historical_status,
        "generation_message": (
            "Loaded from independently validated recovery artifact"
            if len(available) else ""
        ),
    })
dice_case_summary = pd.DataFrame(case_summary_rows)
dice_case_summary.to_csv(ARTIFACTS_DIR / "dice_case_summary.csv", index=False)

dice_final_case_status = pd.read_csv(
    ARTIFACTS_DIR / "dice_final_case_status.csv", keep_default_na=False
)
final_counts = dice_final_case_status["final_status"].value_counts()
assert len(dice_final_case_status) == 20
assert final_counts.get("success_first_pass", 0) == 8
assert final_counts.get("no_counterfactual_returned_after_retry", 0) == 9
assert final_counts.get("timeout_after_retry", 0) == 1
assert final_counts.get("technical_error_first_pass", 0) == 2
display(validated_recovered_counterfactuals[[
    "case_id", "counterfactual_probability",
    "counterfactual_class", "desired_class", "valid_counterfactual"
]])

Recovery generation disabled; loaded saved recovery checkpoints.
Cases expected for recovery: 8
Recovery attempts completed: 8
Counterfactuals returned: 8
Independently valid recovered counterfactuals: 8
Recovery timeouts: 0
Recovery technical errors: 0
Recovery no-CF results: 0
Recovered case IDs: ['XAI_001', 'XAI_002', 'XAI_003', 'XAI_004', 'XAI_005', 'XAI_006', 'XAI_009', 'XAI_017']
All eight cases recovered; standard DiCE artifacts rebuilt.
XAI_005 matches old observed diagnostic: True
Recovered XAI_005 probability=0.418978, AMT_ANNUITY=6187.5
Model retrained: False
Preprocessor refitted: False
Only historical first-pass-success cases regenerated: True


## Deterministic Feature-Change Reconstruction

This non-generative step reconstructs the actionable-feature audit from the persisted recovered counterfactual rows and exact original test rows. Engineered ratios are recomputed for frozen-model validation but are not counted as independent actions.

In [ ]:
ACTIONABLE_FEATURES = ["AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
reconstructed_change_rows = []
for recovered in validated_recovered_counterfactuals.itertuples(index=False):
    original = X_test_raw.loc[int(recovered.row_index), modelling_feature_names]
    for feature in ACTIONABLE_FEATURES:
        original_value = float(original[feature])
        counterfactual_value = float(getattr(recovered, feature))
        changed = not np.isclose(
            original_value, counterfactual_value, equal_nan=True
        )
        absolute_change = abs(counterfactual_value - original_value)
        relative_change = (
            absolute_change / abs(original_value)
            if not np.isclose(original_value, 0.0) else np.nan
        )
        lower, upper = permitted_ranges[feature]
        reconstructed_change_rows.append({
            "case_id": recovered.case_id,
            "counterfactual_id": recovered.counterfactual_id,
            "feature": feature,
            "display_feature": DISPLAY_NAMES.get(feature, feature),
            "original_value": original_value,
            "counterfactual_value": counterfactual_value,
            "absolute_change": absolute_change,
            "relative_change": relative_change,
            "changed": bool(changed),
            "directly_actionable_change": bool(changed),
            "direct_action": bool(changed),
            "at_lower_boundary": bool(np.isclose(counterfactual_value, lower)),
            "at_upper_boundary": bool(np.isclose(counterfactual_value, upper)),
        })
dice_feature_changes = pd.DataFrame(reconstructed_change_rows)
changed_actionable = dice_feature_changes.loc[
    dice_feature_changes["directly_actionable_change"]
]
dice_sparsity_metrics = (
    dice_feature_changes[["case_id", "counterfactual_id"]]
    .drop_duplicates()
    .merge(
        changed_actionable.groupby(["case_id", "counterfactual_id"])
        .size().rename("number_of_changed_actionable_features").reset_index(),
        on=["case_id", "counterfactual_id"], how="left",
    )
)
dice_sparsity_metrics["number_of_changed_actionable_features"] = (
    dice_sparsity_metrics["number_of_changed_actionable_features"]
    .fillna(0).astype(int)
)
dice_sparsity_metrics["number_of_changed_features"] = (
    dice_sparsity_metrics["number_of_changed_actionable_features"]
)
dice_feature_changes.to_csv(ARTIFACTS_DIR / "dice_feature_changes.csv", index=False)
dice_sparsity_metrics.to_csv(ARTIFACTS_DIR / "dice_sparsity_metrics.csv", index=False)

plain_english_rows = []
for counterfactual in dice_counterfactuals.itertuples(index=False):
    changes = dice_feature_changes.loc[
        dice_feature_changes["counterfactual_id"].eq(
            counterfactual.counterfactual_id
        ) & dice_feature_changes["directly_actionable_change"]
    ]
    phrases = [
        f"{row.display_feature.lower()} changed from "
        f"{row.original_value} to {row.counterfactual_value}"
        for row in changes.itertuples()
    ]
    destination = (
        "higher-risk" if counterfactual.counterfactual_class == 1
        else "lower-risk"
    )
    plain_english_rows.append({
        "case_id": counterfactual.case_id,
        "counterfactual_id": counterfactual.counterfactual_id,
        "plain_english_explanation": (
            f"The model assigns the original application a "
            f"{counterfactual.original_probability:.1%} higher-risk probability. "
            f"Under a hypothetical input where {', and '.join(phrases)}, "
            f"the model produces a {counterfactual.counterfactual_probability:.1%} "
            f"higher-risk probability and predicts the {destination} class. "
            "This counterfactual describes model behaviour and is not financial "
            "advice or a causal claim."
        ),
    })
dice_plain_english = pd.DataFrame(plain_english_rows)
dice_plain_english.to_csv(ARTIFACTS_DIR / "dice_plain_english.csv", index=False)

representative = dice_counterfactuals.loc[
    dice_counterfactuals["case_id"].eq("XAI_005")
].iloc[0]
representative_changes = dice_feature_changes.loc[
    dice_feature_changes["case_id"].eq("XAI_005")
    & dice_feature_changes["directly_actionable_change"]
]
positions, width = np.arange(len(representative_changes)), 0.36
plt.figure(figsize=(9, 5))
plt.barh(positions - width / 2, representative_changes["original_value"], height=width, label="Original")
plt.barh(positions + width / 2, representative_changes["counterfactual_value"], height=width, label="Counterfactual")
plt.yticks(positions, representative_changes["display_feature"])
plt.xlabel("Feature value")
plt.title(
    f"DiCE Recovered Counterfactual — XAI_005 | Probability "
    f"{representative['original_probability']:.1%} → "
    f"{representative['counterfactual_probability']:.1%}"
)
plt.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "plots" / "dice_representative_counterfactual.png", dpi=160, bbox_inches="tight")
plt.show()

recovered_case_count = validated_recovered_counterfactuals["case_id"].nunique()
validated_recovered_case_count = int(
    validated_recovered_counterfactuals["valid_counterfactual"].sum()
)
feature_change_row_count = len(dice_feature_changes)
rows_per_case = dice_feature_changes.groupby("case_id").size()
counts = dice_sparsity_metrics["number_of_changed_actionable_features"]
assert RUN_FULL_FIRST_PASS is False
assert RUN_TIMEOUT_RETRY is False
assert RUN_FIRST_PASS_ARTIFACT_RECOVERY is False
assert recovered_case_count == 8
assert validated_recovered_case_count == 8
assert feature_change_row_count == 24
assert rows_per_case.eq(3).all()
assert validated_recovered_counterfactuals["valid_counterfactual"].all()
assert set(dice_feature_changes["feature"]).issubset(ACTIONABLE_FEATURES)
notebook_text = Path("07_dice_counterfactuals.ipynb").read_text(encoding="utf-8")
assert ("model." + "fit(") not in notebook_text
assert ("preprocessor." + "fit(") not in notebook_text
print(f"Recovered counterfactual rows loaded: {recovered_case_count}")
print(f"Independently valid recovered rows: {validated_recovered_case_count}")
print(f"Feature-change rows reconstructed: {feature_change_row_count}")
print(f"Mean changed actionable features: {counts.mean():.3f}")
print(f"Median changed actionable features: {counts.median():.3f}")
print(f"Minimum changed actionable features: {counts.min()}")
print(f"Maximum changed actionable features: {counts.max()}")
print("Representative case: XAI_005")
display(dice_final_case_status[[
    "case_id", "final_status", "valid_counterfactual_available",
    "counterfactual_source"
]])